# Project Continue — Permanent Recovery

Run the cells in order in a **new Google Colab runtime**. This notebook never deletes or overwrites the source backup. It restores the selected backup into a new timestamped folder under `MyDrive/Project-Continue-Recovery-Restores/`.

Source backup root: `MyDrive/Project-Continue-Backups/`. Expected full backup prefix: `recovery_full_`.

**Important:** a runtime-only Agent process cannot be resurrected. Only persisted files/checkpoints found inside the backup can be restored.

In [ ]:
# CELL 1 — Mount Drive and select the newest manifest-bearing full backup
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, datetime, shutil, os

DRIVE_ROOT = Path('/content/drive/MyDrive')
BACKUP_ROOT = DRIVE_ROOT / 'Project-Continue-Backups'
RESTORE_ROOT = DRIVE_ROOT / 'Project-Continue-Recovery-Restores'
if not BACKUP_ROOT.is_dir():
    raise FileNotFoundError(f'Backup root not found: {BACKUP_ROOT}')

candidates = []
for folder in BACKUP_ROOT.glob('recovery_full_*'):
    if not folder.is_dir() or folder.is_symlink():
        continue
    manifests = list(folder.rglob('backup_manifest.json')) + list(folder.rglob('manifest.json'))
    valid_manifest = None
    for manifest in manifests:
        try:
            with manifest.open('r', encoding='utf-8') as f:
                parsed = json.load(f)
            if isinstance(parsed, dict):
                valid_manifest = manifest
                break
        except Exception:
            continue
    if valid_manifest:
        candidates.append((folder.stat().st_mtime, folder, valid_manifest))

if not candidates:
    raise RuntimeError('No recovery_full_* folder with a readable JSON manifest was found. No files were changed.')
candidates.sort(key=lambda item: item[0], reverse=True)
BACKUP_SOURCE = candidates[0][1]
MANIFEST_PATH = candidates[0][2]
print('Selected backup:', BACKUP_SOURCE.name)
print('Manifest:', MANIFEST_PATH.relative_to(BACKUP_SOURCE))
print('Backup timestamp (Drive modified time):', datetime.datetime.fromtimestamp(candidates[0][0]).isoformat())

In [ ]:
# CELL 2 — Restore the entire snapshot to a NEW destination (no overwrite)
RESTORE_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESTORE_DEST = RESTORE_ROOT / f'{BACKUP_SOURCE.name}__RESTORED_{stamp}'
if RESTORE_DEST.exists():
    raise FileExistsError(f'Refusing to overwrite existing restore: {RESTORE_DEST}')

# Copy the complete backup tree, including repository, notebooks, databases,
# agent checkpoints/state files, logs and configuration artifacts if present.
shutil.copytree(BACKUP_SOURCE, RESTORE_DEST, symlinks=False)
print('RESTORE COPY COMPLETE:', RESTORE_DEST)
print('Source backup preserved:', BACKUP_SOURCE.exists())

In [ ]:
# CELL 3 — Verify the restored copy and produce a recovery inventory
assert RESTORE_DEST.is_dir(), 'Restore destination missing'
source_files = [p for p in BACKUP_SOURCE.rglob('*') if p.is_file() and not p.is_symlink()]
restored_files = [p for p in RESTORE_DEST.rglob('*') if p.is_file() and not p.is_symlink()]
source_rel = {p.relative_to(BACKUP_SOURCE) for p in source_files}
restored_rel = {p.relative_to(RESTORE_DEST) for p in restored_files}
missing = sorted(str(p) for p in source_rel - restored_rel)
if missing:
    raise RuntimeError(f'Restore verification failed; missing files: {missing[:20]}')

repo_candidates = [p for p in RESTORE_DEST.rglob('.git') if p.is_dir()]
notebooks = list(RESTORE_DEST.rglob('*.ipynb'))
databases = [p for p in RESTORE_DEST.rglob('*') if p.is_file() and p.suffix.lower() in {'.db','.sqlite','.sqlite3'}]
state_files = [p for p in RESTORE_DEST.rglob('*') if p.is_file() and any(k in p.name.lower() for k in ('agent','state','checkpoint','cursor','session'))]
print('RESTORE VERIFICATION: PASS')
print('Files copied:', len(restored_rel), '| Source files:', len(source_rel))
print('Git repositories:', len(repo_candidates))
print('Notebooks:', len(notebooks))
print('Database files:', len(databases))
print('Agent/state/checkpoint candidates:', len(state_files))
print('Restored location:', RESTORE_DEST)
print('\nThis is a file-level recovery copy. Review the inventory before reconnecting services or starting agents.')

## Manual Final Backup Cell (separate, run only when intentionally creating a new backup)

The recovery cells above do not create or schedule backups. For a new full snapshot, use the existing approved Final Backup Cell from the active Project Continue Colab workflow. Do not run an improvised partial backup here: the canonical cell must include the live repository, databases, notebooks, persisted agent checkpoints and manifest verification. No 15-minute scheduler is installed by this notebook.